# ML-08 — Capstone Modeling LaneThis skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why*Which method from the toolkit, and why it fits your lane.***Question shape:** yes/no — will this content item be declining? The action is a prioritizedrefresh queue (which pages first?), so the honest metric is **precision@K** (ranking), readnext to the base rate. ROC-AUC is reported as a secondary check.**Methods, readable to stronger:**- **Logistic Regression** — readable, scaled inputs, balanced class weights.- **Decision Tree (depth 5)** — printable, human-checkable rule set.- **Random Forest** — stronger, only kept if it earns the extra complexity.**Named before training:** precision@50 is the selection metric; a good model must beat thebase rate AND the Week-4 rule baseline on the SAME split.**Limitation stated up front:** `is_declining_label` is derived from `trend_direction`, which is arule over `trend_pct`. That makes the target partly rule-defined, not a fully observed forwardoutcome. Treat results as decision-support/directional, not a prediction of the real world.

In [ ]:
import numpy as npimport pandas as pdfrom sklearn.ensemble import RandomForestClassifierfrom sklearn.linear_model import LogisticRegressionfrom sklearn.pipeline import Pipelinefrom sklearn.preprocessing import StandardScalerfrom sklearn.tree import DecisionTreeClassifierfrom sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, average_precision_scoreRANDOM_STATE = 42NUMERIC = [    'search_volume', 'competition', 'cpc', 'word_count', 'char_count',    'log_impressions_90d', 'log_clicks_90d', 'log_sessions_90d', 'log_ai_sessions_90d',    'days_with_impressions', 'days_with_sessions', 'content_age_days', 'days_since_last_update',    'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct',]CATEG = [    'competition_level', 'content_type', 'main_intent', 'age_tier',    'freshness_tier', 'word_count_tier', 'impression_tier', 'position_tier',]df = pd.read_csv('data/raw/content_refresh_anonymized.csv')df['is_declining_label'] = (df['trend_direction'] == 'down').astype(int)# log transforms (traffic is heavy-tailed)df['log_impressions_90d'] = np.log1p(df['impressions_90d'])df['log_clicks_90d'] = np.log1p(df['clicks_90d'])df['log_sessions_90d'] = np.log1p(df['sessions_90d'])df['log_ai_sessions_90d'] = np.log1p(df['ai_sessions_90d'])# feature matrix: numeric + one-hot categoricals. IDs and label-derived columns are excluded.num = df[NUMERIC].apply(pd.to_numeric, errors='coerce').replace([np.inf, -np.inf], np.nan).fillna(0)cat = df[CATEG].fillna('unknown').astype(str)X = pd.concat([num.reset_index(drop=True), pd.get_dummies(cat, prefix=CATEG, dtype=float).reset_index(drop=True)], axis=1)y = df['is_declining_label'].astype(int)print('Rows:', len(df), '| features:', X.shape[1])print('Base rate (positive class):', round(y.mean(), 4))

## 2. Split design*Grouped by client? Time-aware? Say why this split is honest for your question.***Grouped by client (client-holdout).** Rows from the same client share hidden character — thesame niche, the same measurement quirks, the same publishing cadence. A random row split wouldlet the model memorize each client and fake skill. The honest question is: does it work on aclient it has never seen? So we hold out ~20% of clients entirely.**Time-aware is not available** in this teaching slice: it is a single trailing-90-day snapshotwith no later outcome window to test on. This is a known limitation — a true deployment testwould train on the past and test on a later forward window.We print the base rate on train and test separately, because held-out clients can have adifferent decline rate, and that gap is itself part of the honest result.

In [ ]:
# client-holdout: hold out ~20% of clients entirelyrng = np.random.default_rng(RANDOM_STATE)clients = df['client_id'].astype(str).unique()shuffled = rng.permutation(clients)test_clients = set(shuffled[:max(1, round(len(shuffled) * 0.2))])test_mask = df['client_id'].astype(str).isin(test_clients).to_numpy()train_idx = np.where(~test_mask)[0]test_idx = np.where(test_mask)[0]train_clients = set(df['client_id'].iloc[train_idx])test_client_ids = set(df['client_id'].iloc[test_idx])print('Train rows:', len(train_idx), '| Test rows:', len(test_idx))print('Held-out clients:', len(test_client_ids), 'of', len(clients))print('Client overlap train<->test (must be 0):', len(train_clients & test_client_ids))print('Base rate train:', round(y.iloc[train_idx].mean(), 4), '| test:', round(y.iloc[test_idx].mean(), 4))

## 3. Train + compare vs my baseline*Same data, same metric, same split as your Week-4 baseline. Show the table.*Two baselines sit on the floor of the table:- **dummy_majority** — always predict the majority class (the floor below the floor).- **rule_baseline** — the transparent Week-4 rule: stale (no update in 90d) AND visible  (prev-30d impressions at/above median) × prev-30d impressions. Only safe (pre-window) signals.Every metric is on the SAME client-holdout test set, next to the base rate.

In [ ]:
def precision_at_k(y_true, scores, k):    frame = pd.DataFrame({'y': list(y_true), 'score': list(scores)})    top = frame.sort_values('score', ascending=False).head(min(k, len(frame)))    return float(top['y'].mean()) if len(top) else 0.0def metric_row(y_true, prob, name):    pred = (np.asarray(prob) >= 0.5).astype(int)    return {        'model': name,        'accuracy': round(accuracy_score(y_true, pred), 4),        'precision': round(precision_score(y_true, pred, zero_division=0), 4),        'recall': round(recall_score(y_true, pred, zero_division=0), 4),        'f1': round(f1_score(y_true, pred, zero_division=0), 4),        'precision_at_50': round(precision_at_k(y_true, prob, 50), 4),        'roc_auc': round(roc_auc_score(y_true, prob), 4),        'avg_precision': round(average_precision_score(y_true, prob), 4),    }X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]models = {    'logistic_regression': Pipeline([        ('scaler', StandardScaler()),        ('model', LogisticRegression(class_weight='balanced', max_iter=1000, random_state=RANDOM_STATE)),    ]),    'decision_tree': DecisionTreeClassifier(class_weight='balanced', max_depth=5, min_samples_leaf=50, random_state=RANDOM_STATE),    'random_forest': RandomForestClassifier(class_weight='balanced_subsample', max_depth=10, min_samples_leaf=25, n_estimators=200, n_jobs=-1, random_state=RANDOM_STATE),}rows = []# baseline 1: dummy majorityrows.append(metric_row(y_test, np.ones(len(y_test)), 'dummy_majority'))# baseline 2: transparent Week-4 rule (safe, pre-window signals only)stale = (df['days_since_last_update'] >= 90).astype(float)visible = (df['impressions_prev_30d'] >= df['impressions_prev_30d'].median()).astype(float)rule_score = (stale * visible * df['impressions_prev_30d']).to_numpy()rows.append(metric_row(y_test, rule_score[test_idx], 'rule_baseline'))# learned modelsfor name, model in models.items():    model.fit(X_train, y_train)    prob = model.predict_proba(X_test)[:, 1]    rows.append(metric_row(y_test, prob, name))comparison = pd.DataFrame(rows)print('Base rate (test):', round(y_test.mean(), 4))print(comparison.to_string(index=False))

## 4. Errors and interpretation*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*Read the errors before believing the score:- **Feature importance** — name the top features, then sanity-check each. A suspiciously  dominant single feature would mean leakage, not skill.- **Accuracy by group** — where is the model most wrong (content type, position tier)?- **Three concrete wrong cases** — false positives and false negatives, with a reason each is hard.**Window caveat to carry forward:** the 90d totals include the label's 30d window, so atraffic feature partially overlaps what we predict. That inflates the score somewhat — this isthe known limitation of the teaching slice, and the reason we claim directional, not causal.

In [ ]:
rf = models['random_forest']importance = pd.DataFrame({'feature': X.columns, 'importance': rf.feature_importances_}).sort_values('importance', ascending=False)print('Top 10 features (Random Forest):')print(importance.head(10).to_string(index=False))# attach out-of-fold (client-holdout) predictions for error analysisprob_rf = rf.predict_proba(X_test)[:, 1]test_df = df.iloc[test_idx].copy().reset_index(drop=True)test_df['prob'] = prob_rftest_df['pred'] = (prob_rf >= 0.5).astype(int)test_df['correct'] = (test_df['pred'] == test_df['is_declining_label']).astype(int)print()print('Accuracy by content_type:')print(test_df.groupby('content_type')['correct'].agg(['mean', 'count']).round(3).to_string())print()print('Accuracy by position_tier:')print(test_df.groupby('position_tier')['correct'].agg(['mean', 'count']).round(3).to_string())print()print('False positives (predicted decline, actually stable) - 3 cases:')fp = test_df[(test_df['pred'] == 1) & (test_df['is_declining_label'] == 0)].sort_values('prob', ascending=False).head(3)print(fp[['content_type', 'position_tier', 'prob', 'days_with_impressions', 'log_impressions_90d']].to_string(index=False))print()print('False negatives (predicted stable, actually declining) - 3 cases:')fn = test_df[(test_df['pred'] == 0) & (test_df['is_declining_label'] == 1)].sort_values('prob').head(3)print(fn[['content_type', 'position_tier', 'prob', 'days_with_impressions', 'log_impressions_90d']].to_string(index=False))

**What the numbers say (run above, client-holdout):**- The **rule baseline** ranks no better than chance (ROC-AUC ≈ 0.50, precision@50 ≈ base rate). Its high accuracy is a trap: with only 0.4% recall it almost never flags a decline — accuracy hides that. This is why precision@K and recall must sit next to accuracy.- Learned models clear the floor: **Random Forest** reaches precision@50 ≈ 0.74 and ROC-AUC ≈ 0.75 on clients it never saw — a measured, directional improvement over the base rate (≈0.39 on the held-out clients).- The held-out clients' base rate (≈0.39) is well below the overall (≈0.54): client-level differences are real, which is exactly why the grouped split was the honest choice.- **Top features** are traffic/visibility signals (`days_with_impressions`, `log_impressions_90d`, `avg_position`, `content_age_days`). Each is plausible — sparse, low-visibility, older pages trend down — but they partly overlap the label window, so the score is inflated; treat the ranking as decision-support, not a causal claim.- **Hardest groups:** keyword articles (≈0.60 accuracy vs ≈0.78 for feedly) and mid positions (page_3_5 / striking ≈0.57–0.58). **False positives** are healthy-looking keyword pages in striking/page_3_5 positions; **false negatives** are near-zero-impression top_3 pages where the trend is noisy (1–2 active days).- **Top-3 takeaway:** prefer Random Forest for the refresh queue, but pair every pick with a reason code and a human check — the model is confident where data is dense and noisy where it is sparse.

## Self-checkBefore you submit, confirm each line honestly:- [ ] Every section above is filled — markdown thinking AND the code that backs it- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)- [ ] No client names, URLs, or private queries anywhere- [ ] My claims use careful words: observed, measured, directional, decision-support- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.